## Testing configuration.

In [1]:
MODEL_PATH   = "jeypiii/Phi-4-mini-instruct_Long_CoT_v3"
DATASET_PATH = "math-ai/amc23"

MAX_SEQ_LENGTH = 3000
LOAD_IN_4BIT   = True
TARGET_TOKEN   = ".answer"

TEMPERATURE = 0.7
TOP_P       = 0.8
TOP_K       = 20
MIN_P       = 0
PRESENCE_P  = 1.5

BATCH_SIZE = 8
TRIALS     = 3

CHAT_TEMPLATE = """\
<|system|>\
You are a fast and responsive reasoning assistant.
Your job is to answer questions and instructions with a concise and to-the-point thought process.<|end|>\
<|user|>\
Solve the math problem below. Respond only with the final numeric answer inside \\boxed{{}}. No solutions or explanations. No units or currencies.

Example Question 1: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Example Answer 1: \\boxed{{72}}

Example Question 2: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Example Answer 2: \\boxed{{10}}

Your Question: {user}
Your Answer: <|end|>\
<|assistant|>\
"""

## Install Unsloth and vLLM.

In [2]:
import os

!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.11.0', 'triton==3.2.0') if is_t4 else ('vllm==vllm==0.11.0', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth wandb==0.24.1 torchcodec==0.5 peft==0.18.1
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.57.6
!uv pip install --no-deps trl==0.22.2
!uv pip uninstall flashinfer-python

os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # Comment this out if out-of-memory
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.5 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 18 packages in 3.77s
Prepared 2 packages in 743ms
Uninstalled 2 packages in 91ms
Installed 2 packages in 64ms
 - huggingface-hub==1.14.0
 + huggingface-hub==0.36.2
 - transformers==5.5.0
 + transformers==4.57.6
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 3ms
Prepared 1 package in 177ms
Uninstalled 1 package in 1ms
Installed 1 package in 5ms
 - trl==0.24.0
 + trl==0.22.2
Using Python 3.12.13 environment at: /usr


## Load the model.

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
  model_name = MODEL_PATH,
  max_seq_length = MAX_SEQ_LENGTH,
  load_in_4bit = LOAD_IN_4BIT,
  fast_inference = True, # Enable vLLM for faster inference
  tensor_parallel_size = 2
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 05-08 10:07:00 [__init__.py:216] Automatically detected platform cuda.
ERROR 05-08 10:07:04 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Phi3 patching. Transformers: 4.57.6. vLLM: 0.11.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
INFO 05-08 10:07:23 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.76.
Unsloth: vLL

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 05-08 10:07:39 [model.py:1733] Casting torch.bfloat16 to torch.float16.
INFO 05-08 10:07:39 [model.py:1510] Using max model len 3000
INFO 05-08 10:07:40 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=4096.
WARNING 05-08 10:07:40 [lora.py:92] `lora_extra_vocab_size` is deprecated and will be removed in v0.12.0. Additional vocabulary support for LoRA adapters is being phased out.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.1.mlp', 'model.layers.3.mlp', 'model.layers.30.mlp'], 'llm_int8_threshold': 6.0}


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

INFO 05-08 10:10:08 [core.py:210] init engine (profile, create kv cache, warmup model) took 103.60 seconds
INFO 05-08 10:10:10 [llm.py:306] Supported_tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['q_norm', 'post_feedforward_layernorm', 'attention_norm', 'layer_norm1', 'pre_feedforward_layernorm', 'k_norm', 'post_attention_layernorm', 'ffn_norm', 'input_layernorm', 'layer_norm2', 'norm1', 'norm2', 'post_layernorm', 'norm']


Some weights of Phi3ForCausalLM were not initialized from the model checkpoint at unsloth/phi-4-mini-instruct-unsloth-bnb-4bit and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['q_norm', 'post_feedforward_layernorm', 'attention_norm', 'cross_attn_input_layernorm', 'layer_norm1', 'pre_feedforward_layernorm', 'k_norm', 'post_attention_layernorm', 'ffn_norm', 'cross_attn_post_attention_layernorm', 'input_layernorm', 'layer_norm2', 'norm1', 'norm2', 'post_layernorm', 'norm']


adapter_model.safetensors:   0%|          | 0.00/35.7M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Phi3ForCausalLM(
      (model): Phi3Model(
        (embed_tokens): Embedding(200064, 3072, padding_idx=200029)
        (layers): ModuleList(
          (0): Phi3DecoderLayer(
            (self_attn): Phi3Attention(
              (o_proj): lora.Linear(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (qkv_proj): Linear4bit(in_fea

## Load the dataset for testing.

In [4]:
from datasets import load_dataset

def formatting_func(batch):
  texts = []
  lengths = []
  for i in range(len(batch["question"])):
    text = CHAT_TEMPLATE.format(user=batch["question"][i])
    texts.append(text)
    lengths.append(len(tokenizer.tokenize(text)))
  return {"text": texts, "length": lengths, "answer": batch["answer"]}

dataset = load_dataset(DATASET_PATH, split="test")
dataset = dataset.map(
  formatting_func,
  batched = True,
  remove_columns = dataset.column_names
)

print(dataset)
print("Max sequence length in dataset:", max(dataset["length"]))

README.md:   0%|          | 0.00/374 [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Dataset({
    features: ['answer', 'text', 'length'],
    num_rows: 40
})
Max sequence length in dataset: 395


In [5]:
print(dataset["text"][0])

<|system|>You are a fast and responsive reasoning assistant.
Your job is to answer questions and instructions with a concise and to-the-point thought process.<|end|><|user|>Solve the math problem below. Respond only with the final numeric answer inside \boxed{}. No solutions or explanations. No units or currencies.

Example Question 1: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Example Answer 1: \boxed{72}

Example Question 2: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Example Answer 2: \boxed{10}

Your Question: Cities $A$ and $B$ are $45$ miles apart. Alicia lives in $A$ and Beth lives in $B$. Alicia bikes towards $B$ at 18 miles per hour. Leaving at the same time, Beth bikes toward $A$ at 12 miles per hour. How many miles from City $A$ will they be when they meet?
Your Answer: <|end|><|assistant|>


## Define vLLM arguments.

In [6]:
from vllm import SamplingParams
from vllm.lora.request import LoRARequest

sampling_params = lambda seed: SamplingParams(
  temperature = TEMPERATURE,
  top_p = TOP_P,
  top_k = TOP_K,
  min_p = MIN_P,
  max_tokens = MAX_SEQ_LENGTH,
  presence_penalty = PRESENCE_P,
  seed = seed,
)

lora_request = LoRARequest("", 1, MODEL_PATH)

## Test evaluation on 1 sample.

In [7]:
import re

prompt = dataset["text"][0]
output = model.fast_generate(
  prompt,
  sampling_params = sampling_params(0),
  lora_request = lora_request,
)[0].outputs[0].text

parts = output.split(TARGET_TOKEN, 1)
answer = parts[-1].strip() if len(parts) > 1 else ""
match = re.search(r'\\boxed\{([^}]*)\}', answer)
answer = match.group(1) if match else ""
output_len = len(tokenizer.tokenize(output))

print(output)
print(">> Final Answer:", answer)
print(">> Output Length:", output_len)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

WARNING 05-08 10:10:43 [processor.py:215] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

.reason
Okay, let's see. The problem is about Alicia and Beth biking towards each other from two cities that are 45 miles apart. They start at the same time, with Alicia going at 18 mph and Beth at 12 mph.

First, I need to figure out how long it takes for them to meet. Since they're moving towards each other, their relative speed is the sum of their individual speeds. So combined, they cover 18 + 12 = 30 miles per hour.

Now, if we think about distance = speed * time, then time = distance / speed. The total distance is 45 miles, and their combined speed is 30 mph. So the time until they meet would be 45 divided by 30. That's 1.5 hours.

Once we know the time, we can find out how far Alicia has traveled in that time. She bikes at 18 mph, so in 1.5 hours, she would go 18 * 1.5. Let's calculate: 18 times 1.5 is 27. So Alicia would have biked 27 miles from City A.

Wait, but maybe I should check. If they meet after 1.5 hours, then together they've covered 30 * 1.5 which is 45 miles, which

## Batched evaluation through the dataset.

In [8]:
import re
import json

for t in range(TRIALS):
  print(f"#### TRIAL {t} ####")

  correct = 0
  output_len = 0
  results = []

  for i in range(0, len(dataset), BATCH_SIZE):
    batchset = dataset.select(range(i, min(i + BATCH_SIZE, len(dataset))))
    prompts = batchset["text"]
    outputs = model.fast_generate(
      prompts,
      sampling_params = sampling_params(t),
      lora_request = lora_request,
    )
    for j, out in enumerate(outputs):
      output = out.outputs[0].text
      parts = output.split(TARGET_TOKEN, 1)
      answer = parts[-1].strip() if len(parts) > 1 else ""
      match = re.search(r'\\boxed\{([^}]*)\}', answer)
      answer = match.group(1) if match else ""
      correct += 1 if (answer == batchset["answer"][j]) else 0
      output_len += len(tokenizer.tokenize(output))
      results += [{
        "correct"      : (answer == batchset["answer"][j]),
        "output_len"   : len(tokenizer.tokenize(output)),
        "prompt"       : prompts[j],
        "ground_truth" : batchset["answer"][j],
        "output"       : output
      }]

    print(f"Batch {(i // BATCH_SIZE) + 1} ({i + len(batchset)} / {len(dataset)}):")
    print(f">> Accuracy      : {correct / (i + len(batchset))}")
    print(f">> Output Length : {output_len / (i + len(batchset))}\n")

  final_stats = {
    "accuracy"        : correct / len(dataset),
    "mean_output_len" : output_len / len(dataset),
  }
  results = [final_stats] + results

  with open(f"[{DATASET_PATH.split("/")[-1].strip()}]_{MODEL_PATH.split("/")[-1].strip()}_(TRIAL {t}).jsonl", "w") as f:
    for item in results:
      f.write(json.dumps(item) + "\n")

#### TRIAL 0 ####


Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Batch 1 (8 / 40):
>> Accuracy      : 0.375
>> Output Length : 855.25



Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

KeyboardInterrupt: 